# Coop Sverige Case — Receipt Data Analysis

Exploration of 2 months of receipt-line data from 2 Coop stores (2020-04-01 to 2020-05-31).

Sections:
1. Load & inspect data
2. Cleaning / derived fields (dates, synthetic item names)
3. Receipt-level (basket) aggregation
4. Channel analysis — online vs. offline vs. omni
5. Sustainability analysis
6. Customer segmentation (MOSAIC / buying power)
7. Category / product exploration


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


## 2. Load data

`rl_2months.csv` is ~1.6M rows. Dtypes are specified up front to keep memory down and avoid
pandas guessing wrong (e.g. treating IDs as floats).

In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)

print(df.shape)
df.head()


## 3. Basic inspection

In [ ]:
df.info(memory_usage="deep")


In [ ]:
# Missing values per column
df.isna().mean().sort_values(ascending=False).to_frame("pct_missing")


In [ ]:
df.describe()


In [ ]:
# Sanity checks on key ID cardinalities (compare against nUnique in case_data_description.xlsx)
for col in ["receiptKey", "customerId", "householdId", "ItemID", "store"]:
    print(f"{col:15s} -> {df[col].nunique():,} unique values")


In [ ]:
# Do the "same % missing" columns actually go missing on the SAME rows?
# (matching percentages alone don't prove that -- confirm directly)
candidate_cols = [
    "customerId", "householdId",
    "MOSAICGroup", "MOSAICGroupDescription",
    "MOSAICType", "MOSAICTypeDescription",
    "DominantBuyingPowerClass",
]
na_masks = df[candidate_cols].isna()

# If True, every column in the group is null on exactly the same rows
same_rows = na_masks.T.drop_duplicates().shape[0] == 1
print("All columns share identical missingness:", same_rows)

# Where these are missing, is it concentrated in one store / channel / date range?
missing_row_mask = na_masks.any(axis=1)
print("\nShare of rows missing these fields:", missing_row_mask.mean())
print("\nBy store:")
print(df.loc[missing_row_mask, "store"].value_counts(normalize=True))
print("\nBy channel (CoopOnlineYN):")
print(df.loc[missing_row_mask, "CoopOnlineYN"].value_counts(normalize=True))


## 4. Derived fields

- Full timestamp from `DayDate` + `hourOfDay` + `minuteOfHour`
- Day of week / weekend flag
- Synthetic item name (per README: no real `ItemName`, so build a proxy from the category hierarchy)
- Net line amount (lineItemAmount is already discount-inclusive; discountAmountExclVat is negative or zero)


In [ ]:
df["timestamp"] = (
    df["DayDate"]
    + pd.to_timedelta(df["hourOfDay"], unit="h")
    + pd.to_timedelta(df["minuteOfHour"], unit="m")
)
df["dayOfWeek"] = df["DayDate"].dt.day_name()
df["isWeekend"] = df["DayDate"].dt.dayofweek >= 5


In [ ]:
# Synthetic item name proxy (see README.txt): concatenate category + segment
df["ItemNameProxy"] = (
    df["ItemCategoryName"].astype(str) + "_" + df["ItemSegmentName"].astype(str)
)
df[["ItemCategoryAreaName", "ItemCategoryGroupName", "ItemCategoryTeamName",
    "ItemCategoryName", "ItemSubCategoryName", "ItemSegmentName",
    "ItemSubSegmentName", "ItemNameProxy"]].drop_duplicates().head(10)


## 5. Receipt-level (basket) aggregation

Line-item data isn't the right grain for "how do customers shop" questions — aggregate up to
one row per receipt first.


In [ ]:
basket = df.groupby("receiptKey", observed=True).agg(
    customerId=("customerId", "first"),
    householdId=("householdId", "first"),
    store=("store", "first"),
    CoopOnlineYN=("CoopOnlineYN", "first"),
    DayDate=("DayDate", "first"),
    dayOfWeek=("dayOfWeek", "first"),
    isWeekend=("isWeekend", "first"),
    hourOfDay=("hourOfDay", "first"),
    n_lines=("ItemID", "count"),
    n_unique_items=("ItemID", "nunique"),
    total_qty=("quantity", "sum"),
    basket_value=("lineItemAmount", "sum"),
    basket_value_excl_vat=("lineItemAmountExclVat", "sum"),
    total_discount=("discountAmountExclVat", "sum"),
).reset_index()

print(basket.shape)
basket.head()


## 6. Channel analysis — online vs. offline

In [ ]:
channel_summary = basket.groupby("CoopOnlineYN", observed=True).agg(
    n_baskets=("receiptKey", "count"),
    avg_basket_value=("basket_value", "mean"),
    median_basket_value=("basket_value", "median"),
    avg_lines_per_basket=("n_lines", "mean"),
    avg_unique_items=("n_unique_items", "mean"),
)
channel_summary


In [ ]:
# Time-of-day pattern by channel
(
    basket.groupby(["CoopOnlineYN", "hourOfDay"], observed=True)
    .size()
    .unstack(level=0)
    .plot(kind="line", figsize=(10, 4), title="Baskets by hour of day, online vs. offline")
)


In [ ]:
# Identify omni-channel customers: households that shopped both online and offline
household_channels = basket.groupby("householdId", observed=True)["CoopOnlineYN"].agg(
    lambda s: set(s)
)
household_channels_summary = household_channels.apply(
    lambda s: "omni" if len(s) > 1 else ("online" if "Y" in s else "offline")
).value_counts()
household_channels_summary


## 7. Sustainability analysis

In [ ]:
sustainability_cols = ["eko", "organic", "krav", "fair_trade", "msc", "no_lactose"]

# Overall share of line items carrying each label
df[sustainability_cols].mean().sort_values(ascending=False).to_frame("share_of_lines")


In [ ]:
# Which categories have the highest share of e.g. organic items?
(
    df.groupby("ItemCategoryTeamName", observed=True)[sustainability_cols]
    .mean()
    .sort_values("organic", ascending=False)
    .head(15)
)


In [ ]:
# Revenue share of sustainably-labelled items vs. not, per label
for col in sustainability_cols:
    rev_share = df.groupby(col, observed=True)["lineItemAmountExclVat"].sum()
    rev_share = rev_share / rev_share.sum()
    print(f"--- {col} ---")
    print(rev_share, "\n")


## 8. Customer segmentation (MOSAIC / buying power)

In [ ]:
# Average basket value by MOSAIC group
seg = basket.merge(
    df[["householdId", "MOSAICGroupDescription", "DominantBuyingPowerClass"]].drop_duplicates("householdId"),
    on="householdId",
    how="left",
)

seg.groupby("MOSAICGroupDescription", observed=True).agg(
    n_baskets=("receiptKey", "count"),
    avg_basket_value=("basket_value", "mean"),
    avg_lines_per_basket=("n_lines", "mean"),
).sort_values("avg_basket_value", ascending=False)


In [ ]:
seg.groupby("DominantBuyingPowerClass", observed=True).agg(
    n_baskets=("receiptKey", "count"),
    avg_basket_value=("basket_value", "mean"),
    avg_lines_per_basket=("n_lines", "mean"),
).sort_values("avg_basket_value", ascending=False)


## 9. Category / product exploration

In [ ]:
# Top categories by revenue
top_categories = (
    df.groupby("ItemCategoryName", observed=True)["lineItemAmountExclVat"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
top_categories


In [ ]:
# Top categories by quantity sold
top_qty = (
    df.groupby("ItemCategoryName", observed=True)["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
top_qty


## 10. Next steps

Ideas to extend this notebook:
- Basket-affinity / market-basket analysis (which categories co-occur in the same receipt)
- Customer lifetime value or repeat-purchase rate by household
- Discount sensitivity by MOSAIC segment or buying-power class
- Weekday vs. weekend category mix
- Store A vs. store B comparison
